# NB: 想定めぐ指数 — 条件不一致重みの最適化

**目的**: 芝↔ダート不一致・距離差 ±600m 超の過去走に対し、加重平均の重みをどれだけ下げるべきかをバックテストで最適化する。

## 戦略

1. **問題**: `megu_final` は過去5走の `megu_index` を `WEIGHTS_B` で単純平均しており、芝走とダート走が混在する。
2. **方針**: 各過去走に条件一致倍率 `m_i ∈ (0,1]` を掛け、正規化した重みで加重平均する。
   - 条件一致 → `w_match = 1.0`
   - 芝/ダのみ不一致 → `w_surface_only`（最適化対象）
   - 距離差 ≥600m のみ → `w_distance_only`（最適化対象）
   - 両方不一致 → `w_both`（最適化対象）
3. **評価**: 2024–2025年の実測 `megu_index` に対する想定値の MAE と、レース内 Spearman 相関の複合スコア `MAE - 5×Spearman` を最小化。
4. **反映**: 最適係数を `config/megu_predict_condition_weights.json` に保存し、`predict_megu_scores()` が自動読込。

## 参照
- `src/pipeline/megu_index/condition_weights.py`
- `src/pipeline/megu_index/optimize_condition_weights.py`
- `docs/decisions/AREA-11-megu-index.md` §6-2

## 0. セットアップ

In [1]:
import sys, os
from pathlib import Path

REPO_ROOT = Path("../../").resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
os.chdir(REPO_ROOT)

from dotenv import load_dotenv
load_dotenv(REPO_ROOT / ".env.stg", override=False)
load_dotenv(REPO_ROOT / ".env", override=False)

import pandas as pd
from sqlalchemy import create_engine

from src.pipeline.megu_index.optimize_condition_weights import (
    fetch_eval_frame,
    evaluate_weights,
    grid_search_condition_weights,
    run_optimization,
    NO_DISCOUNT_WEIGHTS,
)
from src.pipeline.megu_index.condition_weights import load_condition_weights

DATABASE_URL = os.environ.get("DATABASE_URL")
assert DATABASE_URL, "DATABASE_URL が未設定です (.env.stg)"
engine = create_engine(DATABASE_URL)
print("REPO_ROOT:", REPO_ROOT)

REPO_ROOT: /home/jovyan/work/keiba-vpn


## 1. 評価データセット構築

In [2]:
YEAR_START, YEAR_END = 2024, 2025
df_eval = fetch_eval_frame(engine, year_start=YEAR_START, year_end=YEAR_END)
print(f"pairs={len(df_eval):,} races={df_eval['race_id'].nunique():,}")
df_eval.head(3)

pairs=28,403 races=2,995


,race_id,horse_id,actual_megu,surface_target,distance_target,history_json
0,202501010101,2023100344,90.1,芝,1200,"[{'megu_index': 90.2, 'par_time_sec': 70.59, '..."
1,202501010101,2023100672,85.1,芝,1200,"[{'megu_index': 81.1, 'par_time_sec': 70.59, '..."
2,202501010101,2023100772,105.1,芝,1200,"[{'megu_index': 90.4, 'par_time_sec': 69.89, '..."


## 2. ベースライン（重み補正なし相当）

In [3]:
baseline = evaluate_weights(df_eval, NO_DISCOUNT_WEIGHTS)
print(f"Baseline (no discount) MAE={baseline.mae:.3f} RMSE={baseline.rmse:.3f} "
      f"Spearman={baseline.mean_spearman:.3f} n={baseline.n_pairs}")

Baseline (no discount) MAE=11.426 RMSE=14.848 Spearman=0.282 n=28403


## 3. グリッドサーチ最適化

In [4]:
best_cw, grid_df, best_m = grid_search_condition_weights(
    df_eval,
    surface_grid=[0.15, 0.25, 0.35, 0.45, 0.55],
    distance_grid=[0.30, 0.40, 0.50, 0.60, 0.70],
    both_grid=[0.05, 0.10, 0.15, 0.20, 0.25, 0.30],
)
print("Best weights:", best_cw)
print(f"Optimized MAE={best_m.mae:.3f} Spearman={best_m.mean_spearman:.3f} score={best_m.score:.3f}")
grid_df.head(10)

Best weights: {'w_match': 1.0, 'w_surface_only': 0.55, 'w_distance_only': 0.7, 'w_both': 0.3}
Optimized MAE=11.418 Spearman=0.283 score=10.002


,w_surface_only,w_distance_only,w_both,mae,rmse,mean_spearman,score,n_pairs
149,0.55,0.7,0.30,11.418125,14.842900,0.283260,10.001823,28403
143,0.55,0.6,0.30,11.417382,14.843208,0.282982,10.002470,28403
119,0.45,0.7,0.30,11.420118,14.845415,0.283500,10.002620,28403
107,0.45,0.5,0.30,11.419160,14.846861,0.283270,10.002810,28403
137,0.55,0.5,0.30,11.417241,14.844352,0.282879,10.002848,28403
113,0.45,0.6,0.30,11.419318,14.845665,0.283291,10.002864,28403
148,0.55,0.7,0.25,11.418966,14.843740,0.283134,10.003296,28403
142,0.55,0.6,0.25,11.418195,14.844098,0.282912,10.003635,28403
136,0.55,0.5,0.25,11.418093,14.845458,0.282843,10.003880,28403
89,0.35,0.7,0.30,11.423962,14.850753,0.284007,10.003928,28403


## 4. 設定ファイルへ保存

In [5]:
result = run_optimization(engine, year_start=YEAR_START, year_end=YEAR_END, save=True)
print("Saved:", result["config_path"])
print("Loaded back:", load_condition_weights())
result["meta"]

Saved: /home/jovyan/work/keiba-vpn/config/megu_predict_condition_weights.json
Loaded back: {'w_match': 1.0, 'w_surface_only': 0.55, 'w_distance_only': 0.7, 'w_both': 0.3}


{'year_start': 2024,
 'year_end': 2025,
 'baseline': {'mae': 11.42615568777946,
  'mean_spearman': 0.28201845956371174,
  'n_pairs': 28403},
 'default_weights': {'mae': 11.418124845966975,
  'mean_spearman': 0.28326040132418606},
 'optimized': {'mae': 11.418124845966975,
  'rmse': 14.842900262480729,
  'mean_spearman': 0.28326040132418606,
  'score': 10.001822839346044,
  'n_pairs': 28403,
  'n_races': 2995},
 'spearman_weight_in_objective': 5.0}

## 5. 改善幅サマリー

In [6]:
b = result["baseline_metrics"]
o = result["best_metrics"]
summary = pd.DataFrame([
    {"phase": "baseline", "mae": b.mae, "spearman": b.mean_spearman},
    {"phase": "optimized", "mae": o.mae, "spearman": o.mean_spearman},
])
summary["mae_delta"] = summary["mae"] - summary["mae"].iloc[0]
summary["spearman_delta"] = summary["spearman"] - summary["spearman"].iloc[0]
summary

,phase,mae,spearman,mae_delta,spearman_delta
0,baseline,11.426156,0.282018,0.000000,0.000000
1,optimized,11.418125,0.283260,-0.008031,0.001242
